In [0]:
%sql
use catalog datastore

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS datastore.silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS QUARANTINE

In [0]:
# ---------------------------------------------------------------------------
# Full cleansing pipeline for bronze.customers:
#   1. Remove exact + near-duplicates (trailing whitespace, casing differences)
#   2. Flag missing/invalid emails and phone numbers
#   3. Standardize gender and KYC status (mixed casing, abbreviations, blanks)
#   4. Validate date_of_birth (future or implausibly old dates)
#   5. Standardize registration_date across the multiple formats present
#   6. Flag account_id values shared by more than one customer
# Invalid rows -> quarantine.customers ; clean rows -> silver.customers
# ---------------------------------------------------------------------------
import pyspark.sql.functions as F
from pyspark.sql.types import DateType, TimestampType
from pyspark.sql.window import Window

df_raw = spark.table("bronze.customers")
raw_cols = df_raw.columns

# ---- 1. normalize strings & deduplicate (exact + near-duplicates) ----------
df = df_raw
for field in df.schema.fields:
    if field.dataType.simpleString() == "string":
        c = field.name
        df = df.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))

# dedup key: customer_id case-insensitive, whitespace already collapsed;
# keep the row with the latest created_date per customer
w = Window.partitionBy(F.lower(F.col("customer_id"))).orderBy(F.col("created_date").desc())
df = (
    df.withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

EMAIL_RE = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
PHONE_RE = r"^\+?\d{7,15}$"
DATE_FORMATS = [
    "yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy", "MM-dd-yyyy",
    "yyyy/MM/dd", "dd.MM.yyyy", "yyyyMMdd", "dd-MMM-yyyy", "MMM d, yyyy",
]
MIN_DOB = F.to_date(F.lit("1900-01-01"))  # older than ~120 years = implausible


def find_col(*names):
    """Return the first column that exists in the table, else None."""
    return next((n for n in names if n in df.columns), None)


email_col = find_col("email", "email_address")
phone_col = find_col("phone", "phone_number", "mobile")
gender_col = find_col("gender", "gender_cd")
kyc_col = find_col("kyc_status", "kyc")
dob_col = find_col("date_of_birth", "dob", "birth_date")
reg_col = find_col("registration_date", "registered_on", "registration_dt")
account_col = find_col("account_id", "account_number")

# ---- 2. missing / invalid email & phone ------------------------------------
if email_col:
    e = F.lower(F.trim(F.col(email_col)))
    df = df.withColumn("_email_invalid", e.isNull() | (e == "") | ~e.rlike(EMAIL_RE))
else:
    df = df.withColumn("_email_invalid", F.lit(False))

if phone_col:
    # strip separators (keep '+'), then require 7-15 digits
    p = F.regexp_replace(F.trim(F.col(phone_col)), r"[^\d+]", "")
    df = df.withColumn("_phone_invalid", p.isNull() | (p == "") | ~p.rlike(PHONE_RE))
else:
    df = df.withColumn("_phone_invalid", F.lit(False))

# ---- 3. standardize gender / KYC status -------------------------------------
if gender_col:
    g = F.upper(F.trim(F.col(gender_col)))
    df = df.withColumn(
        gender_col + "_std",
        F.when(g.isin("M", "MALE"), "M")
        .when(g.isin("F", "FEMALE"), "F")
        .otherwise("U"),  # blanks / nulls / unrecognized -> Unknown
    )

if kyc_col:
    k = F.upper(F.trim(F.col(kyc_col)))
    df = df.withColumn(
        kyc_col + "_std",
        F.when(
            k.isin("VERIFIED", "V", "Y", "YES", "APPROVED", "COMPLETE", "COMPLETED", "PASS", "PASSED"),
            "VERIFIED",
        )
        .when(k.isin("PENDING", "P", "PROCESSING", "IN_PROGRESS", "UNDER_REVIEW"), "PENDING")
        .when(
            k.isin("NOT_VERIFIED", "N", "NO", "UNVERIFIED", "FAIL", "FAILED", "REJECTED", "BLOCKED"),
            "NOT_VERIFIED",
        )
        .otherwise("UNKNOWN"),  # blanks / nulls / unrecognized
    )


# ---- 4 & 5. parse dob / registration_date across formats --------------------
def parse_any_date(colname):
    """Handle already-typed date columns and strings in any of DATE_FORMATS."""
    field = next((f for f in df.schema.fields if f.name == colname), None)
    if field and isinstance(field.dataType, (DateType, TimestampType)):
        return F.to_date(F.col(colname))
    c = F.trim(F.col(colname))
    parsed = F.try_to_date(c, "yyyy-MM-dd")
    for fmt in DATE_FORMATS:
        parsed = F.coalesce(parsed, F.try_to_date(c, fmt))
    return parsed


if dob_col:
    df = df.withColumn(dob_col + "_std", parse_any_date(dob_col))
    df = df.withColumn(
        "_dob_invalid",
        F.col(dob_col + "_std").isNull()                  # unparseable / missing
        | (F.col(dob_col + "_std") > F.current_date())    # future birth date
        | (F.col(dob_col + "_std") < MIN_DOB),           # implausibly old
    )
else:
    df = df.withColumn("_dob_invalid", F.lit(False))

if reg_col:
    df = df.withColumn(reg_col + "_std", parse_any_date(reg_col))
    df = df.withColumn(
        "_registration_invalid",
        F.col(reg_col + "_std").isNull() | (F.col(reg_col + "_std") > F.current_date()),
    )
else:
    df = df.withColumn("_registration_invalid", F.lit(False))

# ---- 6. account_id shared by multiple customers -----------------------------
shared_ids = []
if account_col:
    shared_ids = [
        r[0]
        for r in df.filter(F.col(account_col).isNotNull())
        .groupBy(account_col)
        .agg(F.countDistinct("customer_id").alias("customers"))
        .filter("customers > 1")
        .select(account_col)
        .collect()
    ]
    df = df.withColumn(
        "_account_invalid",
        F.col(account_col).isNull() | F.col(account_col).isin(shared_ids),
    )
else:
    df = df.withColumn("_account_invalid", F.lit(False))

# ---- combine flags into a human-readable quarantine reason ------------------
df = df.withColumn(
    "quarantine_reasons",
    F.concat_ws(
        "; ",
        F.array(
            F.when(F.col("_email_invalid"), F.lit("missing/invalid email")),
            F.when(F.col("_phone_invalid"), F.lit("missing/invalid phone")),
            F.when(F.col("_dob_invalid"), F.lit("invalid date_of_birth (unparseable/future/implausible)")),
            F.when(F.col("_registration_invalid"), F.lit("unparseable or future registration_date")),
            F.when(F.col("_account_invalid"), F.lit("missing account_id or account_id shared by multiple customers")),
        ),
    ),
)

flagged = df.filter(F.col("quarantine_reasons") != "")
clean = df.filter(F.col("quarantine_reasons") == "")

# standardized values replace the raw ones in the clean set
clean = clean.drop(*[c for c in clean.columns if c.startswith("_")], "quarantine_reasons")
for c in (gender_col, kyc_col, dob_col, reg_col):
    if c and (c + "_std") in clean.columns:
        clean = clean.drop(c).withColumnRenamed(c + "_std", c)
clean = clean.select(*raw_cols)  # restore original column order

# ---- write results -------------------------------------------------------------
quarantine_out = flagged.select(*raw_cols, "quarantine_reasons")
quarantine_out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("quarantine.customers")
clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.customers")

print(f"Raw rows read:            {df_raw.count()}")
print(f"Rows after dedup:         {df.count()}")
print(f"Near-duplicates removed:  {df_raw.count() - df.count()}")
print(f"Quarantined rows:         {flagged.count()}  -> quarantine.customers")
print(f"Clean rows written:       {clean.count()}  -> silver.customers")
if shared_ids:
    print(f"Shared account_ids (quarantined for resolution): {shared_ids}")
if flagged.count() > 0:
    print("\nQuarantine reason breakdown:")
    flagged.groupBy("quarantine_reasons").count().orderBy(F.desc("count")).show(truncate=False)


In [0]:
%sql
select * from silver.customers

In [0]:
%sql
select * from bronze.customers where customer_id='C064'

In [0]:
# ---------------------------------------------------------------------------
# Full cleansing pipeline for bronze.products:
#   1. Remove exact + near-duplicates on product_id (casing/whitespace variants)
#   2. Reject negative, unparseable, or missing prices
#   3. Standardize category casing (mixed case -> Title Case)
#   4. Normalize is_active to a single boolean representation
#      (arrives as boolean, string like 'YES'/'0'/'active', missing, or null)
# Invalid rows -> quarantine.products ; clean rows -> silver.products
# ---------------------------------------------------------------------------
import pyspark.sql.functions as F
from pyspark.sql.types import BooleanType
from pyspark.sql.window import Window

df_raw = spark.table("bronze.products")
raw_cols = df_raw.columns

# ---- 1. normalize strings & deduplicate on product_id ----------------------
df = df_raw
for field in df.schema.fields:
    if field.dataType.simpleString() == "string":
        c = field.name
        df = df.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))

# dedup key: product_id case-insensitive (whitespace already collapsed);
# keep the most recently created row per product when a timestamp exists
order_col = next((c for c in ("created_date", "updated_at", "last_updated") if c in df.columns), None)
w = Window.partitionBy(F.lower(F.col("product_id"))).orderBy(
    F.col(order_col).desc() if order_col else F.monotonically_increasing_id().desc()
)
df = (
    df.withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


def find_col(*names):
    """Return the first column that exists in the table, else None."""
    return next((n for n in names if n in df.columns), None)


price_col = find_col("price", "unit_price", "product_price", "list_price")
category_col = find_col("category", "product_category", "prod_category")
active_col = find_col("is_active", "active", "product_active", "status")

# ---- 2. price: cast to double; reject negative / missing / unparseable -----
# (if you'd rather correct than reject, swap the flag for abs() in silver)
if price_col:
    p = F.col(price_col).cast("double")
    df = df.withColumn(price_col + "_std", p).withColumn(
        "_price_invalid",
        F.col(price_col).isNull() | p.isNull() | (p < 0),
    )
else:
    df = df.withColumn("_price_invalid", F.lit(False))

# ---- 3. category: standardize casing to Title Case ---------------------------
if category_col:
    c = F.col(category_col)
    df = df.withColumn(
        category_col + "_std",
        F.when(c.isNull() | (F.trim(c) == ""), None).otherwise(F.initcap(F.lower(c))),
    )

# ---- 4. is_active: normalize boolean / string / missing / null -> boolean ---
active_out = active_col if active_col else "is_active"  # column created if absent
if active_col:
    field = next((f for f in df.schema.fields if f.name == active_col), None)
    if isinstance(field.dataType, BooleanType):
        base = F.col(active_col)
    else:
        s = F.lower(F.trim(F.col(active_col).cast("string")))
        base = (
            F.when(s.isin("true", "t", "yes", "y", "1", "active", "enabled"), F.lit(True))
            .when(s.isin("false", "f", "no", "n", "0", "inactive", "disabled"), F.lit(False))
            .otherwise(None)
        )
else:
    base = F.lit(None)  # column missing entirely
# unknown (null/missing/unrecognized) -> False (inactive) so the column is a true boolean
df = df.withColumn(active_out + "_std", F.coalesce(base, F.lit(False)))

# ---- combine flags into a human-readable quarantine reason ------------------
df = df.withColumn(
    "quarantine_reasons",
    F.concat_ws(
        "; ",
        F.array(
            F.when(F.col("_price_invalid"), F.lit("missing, unparseable, or negative price")),
        ),
    ),
)

flagged = df.filter(F.col("quarantine_reasons") != "")
clean = df.filter(F.col("quarantine_reasons") == "")

# standardized values replace the raw ones in the clean set
clean = clean.drop(*[c for c in clean.columns if c.startswith("_")], "quarantine_reasons")
for c in (price_col, category_col, active_out):
    if c and (c + "_std") in clean.columns:
        if c in clean.columns:
            clean = clean.drop(c)
        clean = clean.withColumnRenamed(c + "_std", c)
out_cols = raw_cols + ([active_out] if active_out not in raw_cols else [])
clean = clean.select(*out_cols)  # original column order (plus is_active if added)

# ---- write results -------------------------------------------------------------
quarantine_out = flagged.select(*raw_cols, "quarantine_reasons")
quarantine_out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("quarantine.products")
clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.products")

print(f"Raw rows read:            {df_raw.count()}")
print(f"Rows after dedup:         {df.count()}")
print(f"Near-duplicates removed:  {df_raw.count() - df.count()}")
print(f"Quarantined rows:         {flagged.count()}  -> quarantine.products")
print(f"Clean rows written:       {clean.count()}  -> silver.products")
if flagged.count() > 0:
    print("\nQuarantine reason breakdown:")
    flagged.groupBy("quarantine_reasons").count().orderBy(F.desc("count")).show(truncate=False)

In [0]:
# ---------------------------------------------------------------------------
# Full cleansing pipeline for bronze.branches:
#   1. Deduplicate re-sent branch records (same branch_id / near-identical rows)
#   2. Fill or flag missing manager names
#   3. Standardize region and branch_type values (mixed casing, abbreviations, blanks)
#   4. Validate contact number format (digits, length, separators)
# Invalid rows -> quarantine.branches ; clean rows -> silver.branches
# ---------------------------------------------------------------------------
import pyspark.sql.functions as F
from pyspark.sql.window import Window

df_raw = spark.table("bronze.branches")
raw_cols = df_raw.columns

# ---- 1. normalize strings & deduplicate re-sent records --------------------
df = df_raw
for field in df.schema.fields:
    if field.dataType.simpleString() == "string":
        c = field.name
        df = df.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))

# dedup key: branch_id case-insensitive (whitespace already collapsed);
# keep the most recently created/re-sent row per branch when a timestamp exists
order_col = next((c for c in ("created_date", "updated_at", "last_updated", "load_ts") if c in df.columns), None)
w = Window.partitionBy(F.lower(F.col("branch_id"))).orderBy(
    F.col(order_col).desc() if order_col else F.monotonically_increasing_id().desc()
)
df = (
    df.withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


def find_col(*names):
    """Return the first column that exists in the table, else None."""
    return next((n for n in names if n in df.columns), None)


manager_col = find_col("manager_name", "manager", "branch_manager")
region_col = find_col("region", "branch_region", "reg")
type_col = find_col("branch_type", "type", "branch_category")
contact_col = find_col("contact_number", "contact", "phone", "phone_number")

# ---- 2. missing manager names: fill with placeholder + flag for visibility ---
if manager_col:
    m = F.trim(F.col(manager_col))
    df = df.withColumn("_manager_missing", m.isNull() | (m == ""))
    # fill the gap in-place so silver always carries a usable value
    df = df.withColumn(manager_col, F.when(F.col("_manager_missing"), F.lit("UNASSIGNED")).otherwise(m))
else:
    df = df.withColumn("_manager_missing", F.lit(False))

# ---- 3. standardize region (Title Case) and branch_type --------------------
if region_col:
    r = F.col(region_col)
    df = df.withColumn(
        region_col,
        F.when(r.isNull() | (F.trim(r) == ""), F.lit(None)).otherwise(F.initcap(F.lower(r))),
    )

if type_col:
    t = F.upper(F.trim(F.col(type_col)))
    df = df.withColumn(
        type_col,
        F.when(t.isin("BR", "BRANCH", "FULL", "FULL_SERVICE", "MAIN"), "FULL_SERVICE")
        .when(t.isin("ATM", "ATM_ONLY", "CASHPOINT", "SELF_SERVICE"), "ATM_ONLY")
        .when(t.isin("KIOSK", "EXPRESS", "MINI", "MICRO"), "EXPRESS")
        .when(t.isin("HUB", "REGional hub".upper()), "HUB")
        .when(t.isin("", "UNKNOWN"), None)  # blanks / placeholders -> null
        .otherwise(t),  # keep recognized-but-unmapped labels as upper-case
    )

# ---- 4. contact number: strip separators, require 7-15 digits (optional +) ---
CONTACT_RE = r"^\+?\d{7,15}$"
if contact_col:
    # keep '+' and digits only, then validate length/format
    cnum = F.regexp_replace(F.trim(F.col(contact_col)), r"[^\d+]", "")
    df = df.withColumn("_contact_invalid", cnum.isNull() | (cnum == "") | ~cnum.rlike(CONTACT_RE))
    df = df.withColumn(contact_col, cnum)  # store the stripped (standardized) value
else:
    df = df.withColumn("_contact_invalid", F.lit(False))

# ---- combine flags into a human-readable quarantine reason ------------------
df = df.withColumn(
    "quarantine_reasons",
    F.concat_ws(
        "; ",
        F.array(
            F.when(F.col("_contact_invalid"), F.lit("missing or invalidly formatted contact number")),
        ),
    ),
)

flagged = df.filter(F.col("quarantine_reasons") != "")
clean = df.filter(F.col("quarantine_reasons") == "")

# standardized values already replaced the raw ones in-place
clean = clean.drop(*[c for c in clean.columns if c.startswith("_")], "quarantine_reasons")
clean = clean.select(*raw_cols)  # restore original column order

# ---- write results -------------------------------------------------------------
quarantine_out = flagged.select(*raw_cols, "quarantine_reasons")
quarantine_out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("quarantine.branches")
clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.branches")

print(f"Raw rows read:            {df_raw.count()}")
print(f"Rows after dedup:         {df.count()}")
print(f"Re-sent duplicates removed: {df_raw.count() - df.count()}")
print(f"Quarantined rows:         {flagged.count()}  -> quarantine.branches")
print(f"Clean rows written:       {clean.count()}  -> silver.branches")
unassigned = clean.filter(F.col(manager_col) == "UNASSIGNED").count() if manager_col else 0
print(f"Managers filled with UNASSIGNED: {unassigned}")
if flagged.count() > 0:
    print("\nQuarantine reason breakdown:")
    flagged.groupBy("quarantine_reasons").count().orderBy(F.desc("count")).show(truncate=False)
